# **Phase 2 — Build the Donor-Pool Panel**

Standardize match data from the five European leagues into a common structure for synthetic control.
### **Step 1: Standardize League Schedules**

In [1]:
import pandas as pd
from pathlib import Path

RAW_DATA = Path("../data/raw")

la_liga = pd.read_csv(RAW_DATA / "la_liga_2025_2026_schedule.csv")
bundesliga = pd.read_csv(RAW_DATA / "bundesliga_2025_2026_schedule.csv")
premier_league = pd.read_csv(RAW_DATA / "premier_league_2025_2026_schedule.csv")
ligue_1 = pd.read_csv(RAW_DATA / "ligue_1_2025_2026_schedule.csv")
serie_a = pd.read_csv(RAW_DATA / "serie_a_2025_2026_schedule.csv")

print("All league schedules loaded successfully!")

All league schedules loaded successfully!


#### Step 1.1: Verify Loaded Schedules

Checks the size of each league schedule after loading the raw data.

In [2]:
print("La Liga:", la_liga.shape)
print("Bundesliga:", bundesliga.shape)
print("Premier League:", premier_league.shape)
print("Ligue 1:", ligue_1.shape)
print("Serie A:", serie_a.shape)

La Liga: (380, 16)
Bundesliga: (308, 17)
Premier League: (380, 16)
Ligue 1: (308, 17)
Serie A: (380, 16)


### **Step 2: Standardize League Schedules**

Converts each league schedule into a common structure for analysis.

In [4]:
def standardize_schedule(schedule, league_name):
    df = schedule.copy()

    # Convert date to datetime
    df["date"] = pd.to_datetime(df["date"])

    # Create home team records
    home = df[["date", "week", "home_team", "score"]].copy()
    home = home.rename(columns={"home_team": "team"})

    # Create away team records
    away = df[["date", "week", "away_team", "score"]].copy()
    away = away.rename(columns={"away_team": "team"})

    # Calculate points for each match
    def get_points(row, is_home):
        if pd.isna(row["score"]):
            return None

        score = str(row["score"]).split("–")

        if len(score) != 2:
            return None

        try:
            home_score = int(score[0].strip())
            away_score = int(score[1].strip())
        except ValueError:
            return None

        if home_score == away_score:
            return 1

        if is_home:
            return 3 if home_score > away_score else 0

        return 3 if away_score > home_score else 0

    home["points"] = home.apply(
        lambda row: get_points(row, True), axis=1
    )

    away["points"] = away.apply(
        lambda row: get_points(row, False), axis=1
    )

    # Combine home and away records
    standardized = pd.concat([home, away], ignore_index=True)

    # Add league name
    standardized["league"] = league_name

    # Rename week column
    standardized = standardized.rename(
        columns={"week": "matchday"}
    )

    # Keep only required columns
    standardized = standardized[
        ["team", "date", "matchday", "points", "league"]
    ]

    # Sort by team and date
    standardized = standardized.sort_values(
        ["team", "date"]
    ).reset_index(drop=True)

    return standardized

#### Step 2.1: Standardize League Schedules and Confirm the Standardization

Converts each league schedule into a common team-level format and confirms that the standardization was completed successfully.

In [5]:
la_liga_std = standardize_schedule(la_liga, "La Liga")
bundesliga_std = standardize_schedule(bundesliga, "Bundesliga")
premier_league_std = standardize_schedule(premier_league, "Premier League")
ligue_1_std = standardize_schedule(ligue_1, "Ligue 1")
serie_a_std = standardize_schedule(serie_a, "Serie A")

print("Schedules standardized successfully!")

Schedules standardized successfully!


#### Step 2.2: Display Standardized Schedule

Displays the standardized schedule to verify the team-level format and calculated points.

In [6]:
la_liga_std.head(10)

,team,date,matchday,points,league
0,Alavés,2025-08-16,1,3,La Liga
1,Alavés,2025-08-22,2,0,La Liga
2,Alavés,2025-08-30,3,1,La Liga
3,Alavés,2025-09-13,4,3,La Liga
4,Alavés,2025-09-20,5,0,La Liga
5,Alavés,2025-09-24,6,1,La Liga
6,Alavés,2025-09-27,7,0,La Liga
7,Alavés,2025-10-05,8,3,La Liga
8,Alavés,2025-10-20,9,1,La Liga
9,Alavés,2025-10-26,10,0,La Liga


#### Step 2.3: Verify Standardized Schedules

Checks the structure and size of all standardized league schedules.

In [7]:
print("La Liga:", la_liga_std.shape)
print("Bundesliga:", bundesliga_std.shape)
print("Premier League:", premier_league_std.shape)
print("Ligue 1:", ligue_1_std.shape)
print("Serie A:", serie_a_std.shape)

La Liga: (760, 5)
Bundesliga: (616, 5)
Premier League: (760, 5)
Ligue 1: (616, 5)
Serie A: (760, 5)


### **Step 3: Create Relative Matchday Index**

Creates a common matchday index relative to the January 12, 2026 treatment date.

In [8]:
TREATMENT_DATE = pd.Timestamp("2026-01-12")

print("Real Madrid matches around treatment date:")

print(
    la_liga_std[
        (la_liga_std["team"] == "Real Madrid") &
        (la_liga_std["date"].between(
            "2025-12-01",
            "2026-02-01"
        ))
    ][["team", "date", "matchday", "points"]]
)

Real Madrid matches around treatment date:
            team       date  matchday  points
584  Real Madrid 2025-12-03        19       3
585  Real Madrid 2025-12-07        15       0
586  Real Madrid 2025-12-14        16       3
587  Real Madrid 2025-12-20        17       3
588  Real Madrid 2026-01-04        18       3
589  Real Madrid 2026-01-17        20       3
590  Real Madrid 2026-01-24        21       3
591  Real Madrid 2026-02-01        22       3


#### Step 3.1: Sort Matches Chronologically

Sorts each team's matches by actual match date to account for postponed and rescheduled matches.

In [9]:
def sort_team_matches(df):
    df = df.copy()
    
    df["date"] = pd.to_datetime(df["date"])
    
    df = df.sort_values(
        ["team", "date"]
    ).reset_index(drop=True)
    
    return df

#### Step 3.2: Sort Matches Chronologically

Sorts each team's matches by actual match date to account for postponed and rescheduled matches.

In [10]:
la_liga_std = sort_team_matches(la_liga_std)
bundesliga_std = sort_team_matches(bundesliga_std)
premier_league_std = sort_team_matches(premier_league_std)
ligue_1_std = sort_team_matches(ligue_1_std)
serie_a_std = sort_team_matches(serie_a_std)

print("All schedules sorted chronologically!")

All schedules sorted chronologically!


#### Step 3.3: Verify Real Madrid Matches Around the Treatment Date

Displays Real Madrid's matches around the treatment date to verify the matchday sequence and points.

In [11]:
print(
    la_liga_std[
        (la_liga_std["team"] == "Real Madrid") &
        (la_liga_std["date"].between(
            "2025-11-01",
            "2026-02-01"
        ))
    ][["team", "date", "matchday", "points"]]
)

            team       date  matchday  points
580  Real Madrid 2025-11-01        11       3
581  Real Madrid 2025-11-09        12       1
582  Real Madrid 2025-11-23        13       1
583  Real Madrid 2025-11-30        14       1
584  Real Madrid 2025-12-03        19       3
585  Real Madrid 2025-12-07        15       0
586  Real Madrid 2025-12-14        16       3
587  Real Madrid 2025-12-20        17       3
588  Real Madrid 2026-01-04        18       3
589  Real Madrid 2026-01-17        20       3
590  Real Madrid 2026-01-24        21       3
591  Real Madrid 2026-02-01        22       3


#### Step 3.4: Clean Relative Matchday Index

Assigns each team's matches a relative time index based on the treatment date.

In [12]:
def create_relative_index(df, treatment_date):
    df = df.copy()
    
    df["date"] = pd.to_datetime(df["date"])
    
    # Sort each team's matches by actual date
    df = df.sort_values(
        ["team", "date"]
    ).reset_index(drop=True)
    
    # Create sequential match number for each team
    df["match_number"] = df.groupby("team").cumcount()
    
    # Find the number of matches played before treatment
    before_treatment = (
        df["date"] < treatment_date
    )
    
    treatment_position = (
        df[before_treatment]
        .groupby("team")
        .size()
    )
    
    # Assign relative index
    df["relative_matchday"] = (
        df["match_number"]
        - df["team"].map(treatment_position)
    )
    
    return df

In [13]:
TREATMENT_DATE = pd.Timestamp("2026-01-12")

la_liga_std = create_relative_index(
    la_liga_std,
    TREATMENT_DATE
)

bundesliga_std = create_relative_index(
    bundesliga_std,
    TREATMENT_DATE
)

premier_league_std = create_relative_index(
    premier_league_std,
    TREATMENT_DATE
)

ligue_1_std = create_relative_index(
    ligue_1_std,
    TREATMENT_DATE
)

serie_a_std = create_relative_index(
    serie_a_std,
    TREATMENT_DATE
)

print("Relative matchday index created successfully!")

Relative matchday index created successfully!


#### Step 3.4: Verify Relative Matchday Index

Displays Real Madrid's matches around the treatment date to verify the relative matchday index and points.

In [14]:
print(
    la_liga_std[
        (la_liga_std["team"] == "Real Madrid") &
        (la_liga_std["date"].between(
            "2025-11-01",
            "2026-02-01"
        ))
    ][
        ["team", "date", "matchday",
         "relative_matchday", "points"]
    ]
)

            team       date  matchday  relative_matchday  points
580  Real Madrid 2025-11-01        11                 -9       3
581  Real Madrid 2025-11-09        12                 -8       1
582  Real Madrid 2025-11-23        13                 -7       1
583  Real Madrid 2025-11-30        14                 -6       1
584  Real Madrid 2025-12-03        19                 -5       3
585  Real Madrid 2025-12-07        15                 -4       0
586  Real Madrid 2025-12-14        16                 -3       3
587  Real Madrid 2025-12-20        17                 -2       3
588  Real Madrid 2026-01-04        18                 -1       3
589  Real Madrid 2026-01-17        20                  0       3
590  Real Madrid 2026-01-24        21                  1       3
591  Real Madrid 2026-02-01        22                  2       3


### **Step 3.3: Clean Relative Matchday Data**

Removes the temporary match number used to create the relative matchday index.

In [15]:
la_liga_std = la_liga_std.drop(columns=["match_number"])
bundesliga_std = bundesliga_std.drop(columns=["match_number"])
premier_league_std = premier_league_std.drop(columns=["match_number"])
ligue_1_std = ligue_1_std.drop(columns=["match_number"])
serie_a_std = serie_a_std.drop(columns=["match_number"])

print("Temporary columns removed successfully!")

Temporary columns removed successfully!


### **Step 4: Combine League Schedules**

Combines the standardized league schedules into one dataset for analysis.

In [16]:
all_leagues = pd.concat(
    [
        la_liga_std,
        bundesliga_std,
        premier_league_std,
        ligue_1_std,
        serie_a_std
    ],
    ignore_index=True
)

print("All league schedules combined successfully!")
print("Shape:", all_leagues.shape)

All league schedules combined successfully!
Shape: (3512, 6)


In [17]:
all_leagues.head()

,team,date,matchday,points,league,relative_matchday
0,Alavés,2025-08-16,1.0,3,La Liga,-19.0
1,Alavés,2025-08-22,2.0,0,La Liga,-18.0
2,Alavés,2025-08-30,3.0,1,La Liga,-17.0
3,Alavés,2025-09-13,4.0,3,La Liga,-16.0
4,Alavés,2025-09-20,5.0,0,La Liga,-15.0


In [18]:
all_leagues["league"].value_counts()

league
La Liga           760
Premier League    760
Serie A           760
Bundesliga        616
Ligue 1           616
Name: count, dtype: int64

### **Step 5: Filter Analysis Teams**

Keeps Real Madrid and the selected donor teams for the synthetic control analysis.

In [19]:
ANALYSIS_TEAMS = [
    "Real Madrid",
    "Barcelona",
    "Atlético Madrid",
    "Bayern Munich",
    "Manchester City",
    "Liverpool",
    "PSG",
    "Inter",
    "Arsenal"
]

analysis_panel = all_leagues[
    all_leagues["team"].isin(ANALYSIS_TEAMS)
].copy()

print("Analysis teams filtered successfully!")
print("Shape:", analysis_panel.shape)

Analysis teams filtered successfully!
Shape: (334, 6)


In [20]:
analysis_panel["team"].value_counts()

team
Atlético Madrid    38
Barcelona          38
Real Madrid        38
Arsenal            38
Liverpool          38
Manchester City    38
Inter              38
Bayern Munich      34
PSG                34
Name: count, dtype: int64

In [21]:
sorted(analysis_panel["team"].unique())

['Arsenal',
 'Atlético Madrid',
 'Barcelona',
 'Bayern Munich',
 'Inter',
 'Liverpool',
 'Manchester City',
 'PSG',
 'Real Madrid']

### **Step 6: Check Relative Matchday Range**

Checks the available relative matchday range for each analysis team.

In [23]:
analysis_panel.groupby("team")["relative_matchday"].agg(["min", "max", "count"])

,min,max,count
team,,,
Arsenal,-21.0,16.0,38
Atlético Madrid,-19.0,18.0,38
Barcelona,-19.0,18.0,38
Bayern Munich,-16.0,17.0,34
Inter,-19.0,18.0,38
Liverpool,-21.0,16.0,38
Manchester City,-21.0,16.0,38
PSG,-17.0,16.0,34
Real Madrid,-19.0,18.0,38


### **Step 6.1: Align the Common Matchday Range**

Keeps the relative matchday range shared by all analysis teams.

In [24]:
COMMON_MIN = -16
COMMON_MAX = 16

analysis_panel = analysis_panel[
    analysis_panel["relative_matchday"].between(
        COMMON_MIN,
        COMMON_MAX
    )
].copy()

print("Common matchday range:", COMMON_MIN, "to", COMMON_MAX)
print("Shape:", analysis_panel.shape)

Common matchday range: -16 to 16
Shape: (297, 6)


In [25]:
analysis_panel.groupby("team")["relative_matchday"].agg(["min", "max", "count"])

,min,max,count
team,,,
Arsenal,-16.0,16.0,33
Atlético Madrid,-16.0,16.0,33
Barcelona,-16.0,16.0,33
Bayern Munich,-16.0,16.0,33
Inter,-16.0,16.0,33
Liverpool,-16.0,16.0,33
Manchester City,-16.0,16.0,33
PSG,-16.0,16.0,33
Real Madrid,-16.0,16.0,33


### **Step 7: Calculate Rolling Points Per Game**

Calculates a 5-match rolling points-per-game measure for each analysis team.

In [26]:
ROLLING_WINDOW = 5

analysis_panel = analysis_panel.sort_values(
    ["team", "date"]
).copy()

analysis_panel["rolling_ppg"] = (
    analysis_panel
    .groupby("team")["points"]
    .rolling(
        window=ROLLING_WINDOW,
        min_periods=ROLLING_WINDOW
    )
    .mean()
    .reset_index(level=0, drop=True)
)

print("Rolling PPG calculated successfully!")

Rolling PPG calculated successfully!


In [27]:
analysis_panel[
    analysis_panel["team"] == "Real Madrid"
][
    ["team", "date", "relative_matchday", "points", "rolling_ppg"]
].head(10)

,team,date,relative_matchday,points,rolling_ppg
573,Real Madrid,2025-09-13,-16.0,3,NaN
574,Real Madrid,2025-09-20,-15.0,3,NaN
575,Real Madrid,2025-09-23,-14.0,3,NaN
576,Real Madrid,2025-09-27,-13.0,0,NaN
577,Real Madrid,2025-10-04,-12.0,3,2.4
578,Real Madrid,2025-10-19,-11.0,3,2.4
579,Real Madrid,2025-10-26,-10.0,3,2.4
580,Real Madrid,2025-11-01,-9.0,3,2.4
581,Real Madrid,2025-11-09,-8.0,1,2.6
582,Real Madrid,2025-11-23,-7.0,1,2.2


### **Step 7.1: Check Rolling PPG Values**

Checks for missing rolling PPG values after applying the 5-match window.

In [28]:
analysis_panel["rolling_ppg"].isna().sum()

np.int64(36)

### **Step 7.2: Remove Incomplete Rolling PPG Values**

Removes observations without a complete 5-match rolling PPG calculation.

In [29]:
analysis_panel = analysis_panel.dropna(
    subset=["rolling_ppg"]
).copy()

print("Missing rolling PPG values:", analysis_panel["rolling_ppg"].isna().sum())
print("Shape:", analysis_panel.shape)

Missing rolling PPG values: 0
Shape: (261, 7)


In [30]:
analysis_panel.groupby("team").size()

team
Arsenal            29
Atlético Madrid    29
Barcelona          29
Bayern Munich      29
Inter              29
Liverpool          29
Manchester City    29
PSG                29
Real Madrid        29
dtype: int64

### **Step 8: Create the Final Analysis Panel**

Creates the panel structure used for synthetic control analysis.

In [31]:
final_panel = analysis_panel[
    [
        "team",
        "relative_matchday",
        "rolling_ppg",
        "date",
        "league"
    ]
].copy()

final_panel = final_panel.rename(
    columns={
        "team": "unit",
        "relative_matchday": "time",
        "rolling_ppg": "outcome"
    }
)

final_panel = final_panel.sort_values(
    ["unit", "time"]
).reset_index(drop=True)

print("Final analysis panel created!")
print("Shape:", final_panel.shape)

Final analysis panel created!
Shape: (261, 5)


In [32]:
final_panel.head(10)

,unit,time,outcome,date,league
0,Arsenal,-12.0,3.0,2025-11-01,Premier League
1,Arsenal,-11.0,2.6,2025-11-08,Premier League
2,Arsenal,-10.0,2.6,2025-11-23,Premier League
3,Arsenal,-9.0,2.2,2025-11-30,Premier League
4,Arsenal,-8.0,2.2,2025-12-03,Premier League
5,Arsenal,-7.0,1.6,2025-12-06,Premier League
6,Arsenal,-6.0,2.0,2025-12-13,Premier League
7,Arsenal,-5.0,2.0,2025-12-20,Premier League
8,Arsenal,-4.0,2.4,2025-12-27,Premier League
9,Arsenal,-3.0,2.4,2025-12-30,Premier League


### **Step 8.1: Verify Final Panel Balance**

Checks that every analysis team has the same time range and number of observations.

In [33]:
print(
    final_panel.groupby("unit")["time"]
    .agg(["min", "max", "count"])
)
print("Unique time points:", final_panel["time"].nunique())
print("Total observations:", len(final_panel))

                  min   max  count
unit                              
Arsenal         -12.0  16.0     29
Atlético Madrid -12.0  16.0     29
Barcelona       -12.0  16.0     29
Bayern Munich   -12.0  16.0     29
Inter           -12.0  16.0     29
Liverpool       -12.0  16.0     29
Manchester City -12.0  16.0     29
PSG             -12.0  16.0     29
Real Madrid     -12.0  16.0     29
Unique time points: 29
Total observations: 261


### **Step 9: Save Processed Panel**

Saves the final balanced panel for synthetic control analysis.

In [34]:
from pathlib import Path

PROCESSED_DATA = Path("../data/processed")
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

panel_path = PROCESSED_DATA / "panel.csv"

final_panel.to_csv(panel_path, index=False)

print("Panel saved successfully!")
print("Path:", panel_path)

Panel saved successfully!
Path: ..\data\processed\panel.csv


In [35]:
print("Panel shape:", final_panel.shape)
print("Saved file exists:", panel_path.exists())

Panel shape: (261, 5)
Saved file exists: True
